Complementary reading:

- https://www.databricks.com/blog/pdfs-production-announcing-state-art-document-intelligence-databricks
- https://learn.microsoft.com/en-us/azure/databricks/large-language-models/ai-functions
- https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/create-agent

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructType, ArrayType, MapType

# Extraction
import re
import html

# Masking Library
from faker import Faker
import hashlib

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.poc_vector_search

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.poc_vector_search.landing

In [0]:
# Update the following path to your desired location
df = spark.read.format("binaryFile").load("/Volumes/workspace/poc_vector_search/landing/9787565288442.pdf")
df = df.withColumn("parsed_content", F.expr("ai_parse_document(content)"))
df_extract = df.withColumn("content_flat", F.explode(F.expr("try_cast(parsed_content:document:elements AS ARRAY<VARIANT>)"))) \
            .select("path", "content_flat")
display(df_extract)

In [0]:
def quote_col_name(name: str) -> str:
    return f"`{name}`"


def flatten_json(df, col_name, prefix="", stop_flatten_cols=None):
    """
    stop_flatten_cols:
      columns to keep as-is once created, e.g. {"tag_bbox_coord"}
    """
    stop_flatten_cols = set(stop_flatten_cols or [])
    queue = [(col_name, prefix)]

    while queue:
        current_col, current_prefix = queue.pop(0)

        if current_col not in df.columns:
            continue

        if current_col in stop_flatten_cols:
            continue

        dtype = df.schema[current_col].dataType

        if isinstance(dtype, StructType):
            for field in dtype.fields:
                new_col = f"{current_prefix}{field.name}"

                df = df.withColumn(
                    new_col,
                    F.col(
                        f"{quote_col_name(current_col)}.{quote_col_name(field.name)}"
                    )
                )

                if (
                    isinstance(field.dataType, (StructType, ArrayType))
                    and new_col not in stop_flatten_cols
                ):
                    queue.append((new_col, f"{new_col}_"))

            df = df.drop(current_col)

        elif isinstance(dtype, ArrayType):
            df = df.withColumn(
                current_col,
                F.explode_outer(F.col(quote_col_name(current_col)))
            )

            if current_col not in stop_flatten_cols:
                queue.append((current_col, f"{current_col}_"))

    return df


In [0]:
df_json = df_extract.withColumn(
    "content_flat_json",
    F.col("content_flat").cast("string")
)

sample_json = (
    df_json
    .select("content_flat_json")
    .where(F.col("content_flat_json").isNotNull())
    .limit(1)
    .collect()[0]["content_flat_json"]
)

json_schema = F.schema_of_json(F.lit(sample_json))

df_parsed = (
    df_json
    .withColumn(
        "content_flat",
        F.from_json(F.col("content_flat_json"), json_schema)
    )
    .drop("content_flat_json")
)

df_flattened = flatten_json(
    df_parsed,
    "content_flat",
    prefix="tag_",
    stop_flatten_cols={"tag_bbox_coord"}
)

display(df_flattened)

# Extraction of Information

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import MapType, StringType
import re
import html


def extract_payment_block_value(full_text, start_labels, value_label, value_pattern):
    start_pattern = "|".join([re.escape(label) for label in start_labels])

    block_match = re.search(
        rf"({start_pattern})(.*?)(?=Direct debit|EFT|BPAY|BPA|Centrepay|Post Billpay|Credit Card|$)",
        full_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not block_match:
        return None

    block_text = block_match.group(2)

    value_match = re.search(
        rf"{re.escape(value_label)}\s*:?\s*({value_pattern})",
        block_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if value_match:
        return value_match.group(1).strip()

    return None


def extract_relevant_info_from_text(text):
    if text is None:
        return {}

    text = str(text)
    clean_text = html.unescape(text)

    # Convert basic HTML table structure into readable text
    clean_text = re.sub(r"</tr>", "\n", clean_text, flags=re.IGNORECASE)
    clean_text = re.sub(r"</td><td>", " | ", clean_text, flags=re.IGNORECASE)
    clean_text = re.sub(r"</th><th>", " | ", clean_text, flags=re.IGNORECASE)
    clean_text = re.sub(r"<[^>]+>", "", clean_text)

    lines = [
        line.strip()
        for line in clean_text.splitlines()
        if line.strip()
    ]

    full_text = "\n".join(lines)
    info = {}

    # Email
    email_match = re.search(
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
        full_text
    )
    if email_match:
        info["email"] = email_match.group(0)

    # Customer name and address block
    for idx, line in enumerate(lines):
        customer_match = re.match(
            r"^(MR|MRS|MS|MISS|DR)\s+([A-Z]+)(?:\s+([A-Z]+))?\s+([A-Z'-]+)$",
            line,
            flags=re.IGNORECASE
        )

        if customer_match:
            info["customer_title"] = customer_match.group(1).upper()
            info["customer_first_name"] = customer_match.group(2).upper()
            info["customer_middle_name"] = (
                customer_match.group(3).upper()
                if customer_match.group(3)
                else ""
            )
            info["customer_last_name"] = customer_match.group(4).upper()
            info["customer_full_name"] = line.upper()

            # Example: UNIT 409/12 ALBERT ST
            if idx + 1 < len(lines):
                address_line = lines[idx + 1].upper()

                unit_address_match = re.match(
                    r"^(UNIT\s+[A-Z0-9]+)/(.*)$",
                    address_line,
                    flags=re.IGNORECASE
                )

                if unit_address_match:
                    info["property_unit"] = unit_address_match.group(1).strip()
                    info["property_street_address"] = unit_address_match.group(2).strip()
                else:
                    info["property_street_address"] = address_line.strip()

            # Example: HAWTHORN EAST VIC 3123
            if idx + 2 < len(lines):
                suburb_line = lines[idx + 2].upper()

                suburb_match = re.match(
                    r"^([A-Z ]+)\s+(VIC|NSW|QLD|SA|WA|TAS|NT|ACT)\s+(\d{4})$",
                    suburb_line,
                    flags=re.IGNORECASE
                )

                if suburb_match:
                    info["property_suburb"] = suburb_match.group(1).strip()
                    info["property_state"] = suburb_match.group(2).upper()
                    info["property_postcode"] = suburb_match.group(3)

            address_parts = [
                info.get("property_unit"),
                info.get("property_street_address"),
                info.get("property_suburb"),
                info.get("property_state"),
                info.get("property_postcode")
            ]

            info["property_address"] = ", ".join(
                [part for part in address_parts if part]
            )

            break

    # General bill fields
    label_patterns = {
        "account_number": r"Account number\s*(?:\||\s)\s*([0-9 ]{6,})",
        "invoice_number": r"Invoice number\s*(?:\||\s)\s*([0-9 ]{8,})",
        "issue_date": r"Issue date\s*(?:\||\s)\s*([0-9]{1,2}\s+[A-Za-z]{3}\s+[0-9]{4})",
        "property_reference": r"Property reference\s*(?:\||\s)\s*([A-Z0-9, ]+)",
        "total_due": r"Total due\s*(?:\||\s)\s*(\$[0-9,.]+)",
        "direct_debit_date": r"Direct debit\s*(?:\||\s)\s*([0-9]{1,2}\s+[A-Za-z]{3}\s+[0-9]{4})",
        "crn_reference": r"CRN reference:\s*([0-9A-Z ]+)",
    }

    for key, pattern in label_patterns.items():
        match = re.search(pattern, full_text, flags=re.IGNORECASE)
        if match:
            info[key] = match.group(1).strip()

    # Meter number: generic, requires letters + digits
    meter_match = re.search(
        r"Meter number.*?\b([A-Z]{2,}[A-Z0-9\-]*\d+[A-Z0-9\-]*)\b",
        full_text,
        flags=re.IGNORECASE | re.DOTALL
    )
    if meter_match:
        info["meter_number"] = meter_match.group(1).strip()

    # Payment fields from labelled sections
    info["bpay_biller_code"] = extract_payment_block_value(
        full_text,
        ["BPAY", "BPA", "BPAY®"],
        "Biller code",
        r"[0-9]+"
    )

    info["bpay_ref"] = extract_payment_block_value(
        full_text,
        ["BPAY", "BPA", "BPAY®"],
        "Ref",
        r"[0-9 ]{6,}"
    )

    info["post_billpay_biller_code"] = extract_payment_block_value(
        full_text,
        ["Post Billpay", "Post Billpay®"],
        "Biller code",
        r"[0-9]+"
    )

    info["post_billpay_ref"] = extract_payment_block_value(
        full_text,
        ["Post Billpay", "Post Billpay®"],
        "Ref",
        r"[0-9 ]{6,}"
    )

    # Stronger BPAY fallback
    if not info.get("bpay_biller_code"):
        bpay_code_match = re.search(
            r"(?:BPAY|BPA|BPAY®).*?Biller code:\s*([0-9]+)",
            full_text,
            flags=re.IGNORECASE | re.DOTALL
        )
        if bpay_code_match:
            info["bpay_biller_code"] = bpay_code_match.group(1).strip()

    if not info.get("bpay_ref"):
        bpay_ref_match = re.search(
            r"(?:BPAY|BPA|BPAY®).*?Ref:\s*([0-9 ]{6,})",
            full_text,
            flags=re.IGNORECASE | re.DOTALL
        )
        if bpay_ref_match:
            info["bpay_ref"] = bpay_ref_match.group(1).strip()

    # Stronger Post Billpay fallback
    if not info.get("post_billpay_biller_code"):
        post_code_match = re.search(
            r"Post Billpay.*?Biller code:\s*([0-9]+)",
            full_text,
            flags=re.IGNORECASE | re.DOTALL
        )
        if post_code_match:
            info["post_billpay_biller_code"] = post_code_match.group(1).strip()

    if not info.get("post_billpay_ref"):
        post_ref_match = re.search(
            r"Post Billpay.*?Ref:\s*([0-9 ]{6,})",
            full_text,
            flags=re.IGNORECASE | re.DOTALL
        )
        if post_ref_match:
            info["post_billpay_ref"] = post_ref_match.group(1).strip()

    # Layout-specific fallback:
    # For this bill type, Post Billpay ref is usually the invoice number.
    if not info.get("post_billpay_ref") and info.get("invoice_number"):
        info["post_billpay_ref"] = info["invoice_number"]

    # Layout-specific fallback:
    # BPAY ref is often 3-4-4 digit grouping, e.g. 978 5401 5816.
    if not info.get("bpay_ref"):
        bpay_ref_match = re.search(
            r"\b\d{3}\s\d{4}\s\d{4}\b",
            full_text
        )
        if bpay_ref_match:
            info["bpay_ref"] = bpay_ref_match.group(0)

    # Remove empty values
    return {
        key: value
        for key, value in info.items()
        if value is not None and str(value).strip() != ""
    }


extract_relevant_info_from_text_udf = F.udf(
    extract_relevant_info_from_text,
    MapType(StringType(), StringType())
)

In [0]:
extract_relevant_info_from_text_udf = F.udf(
    extract_relevant_info_from_text,
    MapType(StringType(), StringType())
)


# Build one text blob per PDF/path using page + tag order
df_document_text = (
    df_flattened
    .where(F.col("tag_content").isNotNull())
    .groupBy("path")
    .agg(
        F.sort_array(
            F.collect_list(
                F.struct(
                    F.col("tag_bbox_page_id").alias("page_id"),
                    F.col("tag_id").alias("tag_id"),
                    F.col("tag_content").alias("content")
                )
            )
        ).alias("content_items")
    )
    .withColumn(
        "document_text",
        F.expr("concat_ws('\n', transform(content_items, x -> x.content))")
    )
    .drop("content_items")
)


# Extract from full document text, not row-level tag_content
df_info_extracted = (
    df_document_text
    .withColumn(
        "extracted_info",
        extract_relevant_info_from_text_udf(F.col("document_text"))
    )
)

display(df_info_extracted)

In [0]:
# Dynamically expand map keys
extracted_keys = (
    df_info_extracted
    .select(F.explode(F.map_keys("extracted_info")).alias("key"))
    .distinct()
    .orderBy("key")
    .collect()
)

extracted_keys = [row["key"] for row in extracted_keys]


df_extracted_only = (
    df_info_extracted
    .select(
        F.col("path"),
        *[
            F.col("extracted_info").getItem(key).alias(key)
            for key in extracted_keys
        ]
    )
)


# Coalesce duplicate rows by path
# If a field is null in one row but populated in another, keep the populated value
df_extracted_only_coalesced = (
    df_extracted_only
    .groupBy("path")
    .agg(
        *[
            F.first(F.col(key), ignorenulls=True).alias(key)
            for key in extracted_keys
        ]
    )
)


display(df_extracted_only_coalesced)

# Masking

TODO: Plug the below to extracted info df as well

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from faker import Faker
import hashlib
import re


EXCLUDE_FROM_MASKING_PATTERNS = [
    "file",
    "filename",
    "file_name",
    "filepath",
    "file_path",
    "_metadata",
    "path",
    "bbox",
    "confidence",
    "page_id"
]


def should_exclude_from_masking(col_name: str) -> bool:
    col_lower = col_name.lower()
    return any(pattern in col_lower for pattern in EXCLUDE_FROM_MASKING_PATTERNS)


def stable_seed(value: str, context: str = "") -> int:
    raw = f"{context}|{value}"
    return int(hashlib.sha256(raw.encode("utf-8")).hexdigest()[:8], 16)


def fake_for_value(value: str, pii_type: str, context: str = "") -> str:
    fake = Faker("en_AU")
    fake.seed_instance(stable_seed(value, context))

    if pii_type == "email":
        return fake.email()

    if pii_type == "phone":
        return fake.phone_number()

    if pii_type == "name":
        return fake.name()

    if pii_type == "address":
        return fake.street_address().upper()

    if pii_type == "suburb":
        return fake.city().upper()

    if pii_type == "account_number":
        return f"{fake.random_number(digits=2, fix_len=True)} {fake.random_number(digits=4, fix_len=True)} {fake.random_number(digits=4, fix_len=True)}"

    if pii_type == "invoice_number":
        return f"{fake.random_number(digits=4, fix_len=True)} {fake.random_number(digits=4, fix_len=True)} {fake.random_number(digits=5, fix_len=True)}"

    if pii_type == "property_reference":
        return f"{fake.random_number(digits=7, fix_len=True)}, LOT B{fake.random_number(digits=3, fix_len=True)}"

    if pii_type == "meter_number":
        return f"YAAD{fake.random_number(digits=6, fix_len=True)}"

    if pii_type == "crn":
        return f"{fake.random_number(digits=3, fix_len=True)} {fake.random_number(digits=3, fix_len=True)} {fake.random_number(digits=3, fix_len=True)}T"

    if pii_type == "biller_ref":
        return f"{fake.random_number(digits=4, fix_len=True)} {fake.random_number(digits=4, fix_len=True)} {fake.random_number(digits=5, fix_len=True)}"

    return str(value)


def mask_text_value(value):
    if value is None:
        return None

    text = str(value)

    # Email
    text = re.sub(
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
        lambda m: fake_for_value(m.group(0), "email", "email"),
        text
    )

    # Phone numbers, including 1300, 1800, 13 xx xx, mobiles
    text = re.sub(
        r"\b(?:13\s?\d{2}\s?\d{2}|1300\s?\d{3}\s?\d{3}|1800\s?\d{3}\s?\d{3}|04\d{2}\s?\d{3}\s?\d{3})\b",
        lambda m: fake_for_value(m.group(0), "phone", "phone"),
        text
    )

    # Account number labels
    text = re.sub(
        r"(?i)(Account number</td><td>)([^<]+)",
        lambda m: m.group(1) + fake_for_value(m.group(2), "account_number", "account_number"),
        text
    )

    text = re.sub(
        r"(?i)(Account number:\s*)([0-9 ]{6,})",
        lambda m: m.group(1) + fake_for_value(m.group(2), "account_number", "account_number"),
        text
    )

    # Invoice number labels
    text = re.sub(
        r"(?i)(Invoice number</td><td>)([^<]+)",
        lambda m: m.group(1) + fake_for_value(m.group(2), "invoice_number", "invoice_number"),
        text
    )

    # Property reference
    text = re.sub(
        r"(?i)(Property reference</td><td>)([^<]+)",
        lambda m: m.group(1) + fake_for_value(m.group(2), "property_reference", "property_reference"),
        text
    )

    # Biller / payment refs
    text = re.sub(
        r"(?i)(Ref:\s*)([0-9 ]{6,})",
        lambda m: m.group(1) + fake_for_value(m.group(2), "biller_ref", "biller_ref"),
        text
    )

    text = re.sub(
        r"(?i)(CRN reference:\s*)([0-9A-Z ]{6,})",
        lambda m: m.group(1) + fake_for_value(m.group(2), "crn", "crn"),
        text
    )

    # Meter number like YAAD032385
    text = re.sub(
        r"\bYAAD\d{6}\b",
        lambda m: fake_for_value(m.group(0), "meter_number", "meter_number"),
        text
    )

    # Name pattern like MR C ZHAN
    text = re.sub(
        r"\b(MR|MRS|MS|MISS|DR)\s+[A-Z]\s+[A-Z]{2,}\b",
        lambda m: fake_for_value(m.group(0), "name", "name").upper(),
        text
    )

    # Unit / street address
    text = re.sub(
        r"\bUNIT\s+\d+[A-Z]?/\d+\s+[A-Z ]+\s+(ST|RD|ROAD|AVE|AVENUE|DR|DRIVE|CT|COURT)\b",
        lambda m: "UNIT 101/" + fake_for_value(m.group(0), "address", "address"),
        text
    )

    text = re.sub(
        r"\b\d+\s+[A-Z ]+\s+(ST|RD|ROAD|AVE|AVENUE|DR|DRIVE|CT|COURT)\b",
        lambda m: fake_for_value(m.group(0), "address", "address"),
        text
    )

    # Postcode-like suburb/state/postcode
    text = re.sub(
        r"\b[A-Z][A-Z ]+\s+VIC\s+\d{4}\b",
        lambda m: fake_for_value(m.group(0), "suburb", "suburb") + " VIC 3000",
        text
    )

    return text


mask_text_udf = F.udf(mask_text_value, StringType())


def mask_pii_spi_content_aware(df):
    for field in df.schema.fields:
        col_name = field.name

        if should_exclude_from_masking(col_name):
            continue

        if isinstance(field.dataType, StringType):
            df = df.withColumn(
                col_name,
                mask_text_udf(F.col(col_name))
            )

    return df


df_masked = mask_pii_spi_content_aware(df_flattened)

display(df_masked)